In [ ]:
import re
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [ ]:
df = pd.read_csv("../final_result/hard_indiv_for_figure.csv")

In [ ]:
def build_edges_from_df(df):
    """
    Create an edge dataframe with columns:
        source, target, weight, error_rate

    Expected df columns:
        Individual_ID
        Error_Rate
        Confusion_Details

    Example Confusion_Details:
        "FAC002 (75), FAC010 (48)"
    """

    required_columns = {
        "Individual_ID",
        "Error_Rate",
        "Confusion_Details"
    }

    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    edge_rows = []

    for _, row in df.iterrows():
        source = str(row["Individual_ID"])
        error_rate = float(row["Error_Rate"])
        confusion_text = str(row["Confusion_Details"])

        # Extract identities and confusion counts:
        # FAC002 (75) -> target=FAC002, weight=75
        matches = re.findall(
            r"(FAC\d+)\s*\((\d+)\)",
            confusion_text
        )

        for target, weight in matches:
            edge_rows.append({
                "source": source,
                "target": str(target),
                "weight": int(weight),
                "error_rate": error_rate
            })

    return pd.DataFrame(
        edge_rows,
        columns=["source", "target", "weight", "error_rate"]
    )


# Build edges
edges = build_edges_from_df(df)

# Check the result
print(edges.head())
print(edges.columns.tolist())
print(f"Number of edges: {len(edges)}")

In [ ]:
# ==================================================
# PLOTTING SETTINGS
# ==================================================

# Number of highest-error true identities to display
top_n_sources = 12

# Number of strongest confusion targets retained per true identity
top_k_targets = 2

# Exclude confusion relationships below this count
min_edge_weight = 45

# Output image path
SAVE_PATH = "directed_confusion_network_clean.png"


# ==================================================
# COLOR PALETTE
# ==================================================

# Appears only as a true identity/source
SOURCE_COLOR = "#2B6CB0"

# Appears only as a predicted identity/target
TARGET_COLOR = "#F6AD55"

# Appears as both a source and a target
BOTH_COLOR = "#805AD5"

# Arrow color
EDGE_COLOR = "#4A5568"


# ==================================================
# SELECT EDGES TO DISPLAY
# ==================================================

# Select the identities with the highest classification error rates
top_sources = (
    df.sort_values("Error_Rate", ascending=False)
      .head(top_n_sources)["Individual_ID"]
      .astype(str)
      .tolist()
)

# Keep only confusion edges originating from the selected identities
edges_plot = edges[
    edges["source"].astype(str).isin(top_sources)
].copy()

# Keep the strongest outgoing confusion targets for each source
edges_plot = (
    edges_plot
    .sort_values(
        ["source", "weight"],
        ascending=[True, False]
    )
    .groupby("source", as_index=False)
    .head(top_k_targets)
    .reset_index(drop=True)
)

# Remove weak confusion relationships
edges_plot = edges_plot[
    edges_plot["weight"] >= min_edge_weight
].copy()


# ==================================================
# BUILD THE DIRECTED GRAPH
# ==================================================

G = nx.DiGraph()

# Map each individual to its overall classification error rate
error_rate_map = (
    df.assign(
        Individual_ID=df["Individual_ID"].astype(str)
    )
    .set_index("Individual_ID")["Error_Rate"]
    .to_dict()
)

# Add nodes and directed confusion edges
for _, row in edges_plot.iterrows():
    true_id = str(row["source"])
    predicted_id = str(row["target"])
    confusion_count = int(row["weight"])

    # Add the true identity node
    G.add_node(
        true_id,
        error_rate=error_rate_map.get(true_id, 0)
    )

    # Add the predicted identity node
    G.add_node(
        predicted_id,
        error_rate=error_rate_map.get(predicted_id, 0)
    )

    # Edge direction:
    # true identity --> predicted identity
    G.add_edge(
        true_id,
        predicted_id,
        weight=confusion_count
    )


# ==================================================
# DETERMINE NODE ROLES
# ==================================================

# Nodes appearing as true identities
sources_in_graph = set(
    edges_plot["source"].astype(str)
)

# Nodes appearing as predicted identities
targets_in_graph = set(
    edges_plot["target"].astype(str)
)

node_role = {}

for node in G.nodes():
    if node in sources_in_graph and node in targets_in_graph:
        node_role[node] = "both"

    elif node in sources_in_graph:
        node_role[node] = "source"

    else:
        node_role[node] = "target"


# ==================================================
# CALCULATE THE NETWORK LAYOUT
# ==================================================

# Use Graphviz when available because it generally produces a cleaner
# hierarchical layout for directed networks.
try:
    from networkx.drawing.nx_agraph import graphviz_layout

    pos = graphviz_layout(
        G,
        prog="dot"
    )

    # Normalize Graphviz coordinates to the range 0–1
    x_coordinates = np.array([
        position[0]
        for position in pos.values()
    ])

    y_coordinates = np.array([
        position[1]
        for position in pos.values()
    ])

    x_min = x_coordinates.min()
    x_max = x_coordinates.max()
    y_min = y_coordinates.min()
    y_max = y_coordinates.max()

    pos = {
        node: (
            (pos[node][0] - x_min) / (x_max - x_min + 1e-9),
            (pos[node][1] - y_min) / (y_max - y_min + 1e-9)
        )
        for node in pos
    }

except Exception:
    # Fallback layout when Graphviz or pygraphviz is unavailable
    pos = nx.kamada_kawai_layout(
        G,
        weight="weight"
    )


# ==================================================
# DEFINE NODE APPEARANCE
# ==================================================

node_colors = []
node_sizes = []

for node in G.nodes():
    role = node_role[node]
    error_rate = G.nodes[node]["error_rate"]

    # Assign color according to the node's role
    if role == "source":
        node_colors.append(SOURCE_COLOR)

    elif role == "target":
        node_colors.append(TARGET_COLOR)

    else:
        node_colors.append(BOTH_COLOR)

    # Increase node size slightly with classification error
    node_sizes.append(
        1800 + 1000 * error_rate
    )


# ==================================================
# DEFINE EDGE APPEARANCE
# ==================================================

# Use the same thickness for every arrow
edge_width = 1.0


# ==================================================
# CREATE NODE LABELS
# ==================================================

# Each node displays its ID and classification error percentage
node_labels = {
    node: (
        f"{node}\n"
        f"{100 * G.nodes[node]['error_rate']:.1f}%"
    )
    for node in G.nodes()
}


# ==================================================
# DRAW THE NETWORK
# ==================================================

plt.figure(figsize=(12, 9))


# --------------------------------------------------
# Draw directed arrows
# --------------------------------------------------
# Arrows are drawn before the nodes so their endpoints remain visually
# underneath the node boundaries.
nx.draw_networkx_edges(
    G,
    pos,
    width=edge_width,
    edge_color=EDGE_COLOR,
    alpha=0.55,
    arrows=True,
    arrowsize=24,
    arrowstyle="-|>",
    min_source_margin=18,
    min_target_margin=22,
    connectionstyle="arc3,rad=0.10"
)


# --------------------------------------------------
# Draw nodes
# --------------------------------------------------
nx.draw_networkx_nodes(
    G,
    pos,
    node_size=node_sizes,
    node_color=node_colors,
    edgecolors="black",
    linewidths=1.0,
    alpha=0.96
)


# --------------------------------------------------
# Draw node labels
# --------------------------------------------------
nx.draw_networkx_labels(
    G,
    pos,
    labels=node_labels,
    font_size=8,
    font_weight="bold",
    font_color="white"
)


# ==================================================
# LEGEND
# ==================================================

legend_handles = [
    mpatches.Patch(
        color=SOURCE_COLOR,
        label="True identity / source"
    ),
    mpatches.Patch(
        color=TARGET_COLOR,
        label="Predicted identity / misclassified-as"
    ),
    mpatches.Patch(
        color=BOTH_COLOR,
        label="Appears as both source and target"
    )
]

plt.legend(
    handles=legend_handles,
    title="Node type",

    # Place the legend inside the lower-right corner
    loc="lower right",
    bbox_to_anchor=(0.98, 0.02),

    # Display entries vertically
    ncol=1,

    # Remove the surrounding legend frame
    frameon=False,

    # Legend text sizes
    fontsize=13.5,
    title_fontsize=15,

    # Marker and spacing controls
    handlelength=2.0,
    handleheight=1.2,
    labelspacing=0.8
)


# ==================================================
# FINALIZE AND SAVE
# ==================================================

# Remove axes because network coordinates have no quantitative meaning
plt.axis("off")

# Reduce unnecessary whitespace
plt.tight_layout()

# Save as a high-resolution image
plt.savefig(
    SAVE_PATH,
    dpi=300,
    bbox_inches="tight"
)

# Display the figure
plt.show()